# 4. Orquestación de Agentes con CrewAI

## Objetivos de Aprendizaje
- Entender el concepto de agentes colaborativos y los roles en un "equipo" (Crew).
- Aprender los componentes clave de CrewAI: `Agent`, `Task`, `Tool` y `Crew`.
- Definir un equipo de agentes (un investigador y un escritor) para realizar una tarea compleja.
- Ejecutar el `Crew` y observar cómo los agentes colaboran y se pasan el trabajo entre ellos.
- **Comprender las configuraciones específicas necesarias para usar CrewAI con GitHub Models API**.

## ¿Qué es CrewAI y por qué usarlo?

Mientras que LangChain proporciona los bloques de construcción fundamentales para crear un agente, **CrewAI** se especializa en la **orquestación de múltiples agentes autónomos**. La idea central es que, para resolver tareas complejas, es más eficiente tener un equipo de agentes especializados que colaboren, en lugar de un solo agente que intente hacerlo todo.

**Analogía:** Piensa en una agencia de marketing. No tienes una sola persona que es experta en investigación, redacción, diseño y redes sociales. Tienes un equipo donde cada miembro tiene un rol claro. CrewAI aplica este concepto a los agentes de IA.

**Ventajas de CrewAI:**
- **Roles Especializados**: Permite definir agentes con roles, objetivos (`goal`) e historias de fondo (`backstory`) específicas, lo que los hace más efectivos en su nicho.
- **Colaboración Autónoma**: Los agentes pueden delegar tareas entre ellos de forma autónoma.
- **Procesos Secuenciales y Jerárquicos**: Soporta flujos de trabajo donde las tareas se completan en un orden específico.
- **Claridad y Estructura**: El código es muy declarativo y fácil de leer, ya que se centra en definir el equipo y sus responsabilidades.

## Relación entre CrewAI y LangChain

CrewAI **utiliza LangChain internamente** para manejar los LLMs y las herramientas. Esto significa que:

1. **CrewAI** se encarga de la orquestación de agentes (el "director de orquesta")
2. **LangChain** proporciona la interfaz con los modelos y herramientas (los "instrumentos")

Esta relación requiere configuraciones específicas que veremos en este notebook.

### 1. Instalación y Configuración

In [14]:
!pip install crewai crewai-tools langchain-openai openai requests python-dotenv -q

### 2. Configuración Especial para GitHub Models API

**⚠️ IMPORTANTE:** Esta es la parte crítica que causa problemas si no se configura correctamente.

**El Problema:**
- CrewAI utiliza LangChain internamente
- LangChain busca las variables de entorno `OPENAI_API_KEY` y `OPENAI_API_BASE`
- Nosotros tenemos `GITHUB_TOKEN` y `OPENAI_BASE_URL`
- Necesitamos "mapear" nuestras variables a las que LangChain espera

**La Solución:**
Configuramos las variables de entorno que LangChain espera, usando nuestros valores de GitHub Models API.

In [15]:
import os
from pathlib import Path
from dotenv import load_dotenv

for env_path in [Path.cwd() / ".env", *[parent / ".env" for parent in Path.cwd().parents]]:
    if env_path.exists():
        load_dotenv(env_path)
        break

# 🔧 CONFIGURACIÓN CRÍTICA: Mapear variables de entorno para CrewAI
# CrewAI → OpenAI-compatible API → GitHub Models API

base_url = os.environ.get("OPENAI_BASE_URL") or os.environ.get("GITHUB_BASE_URL", "")
github_token = os.environ.get("GITHUB_TOKEN", "")

# LangChain/OpenAI pueden leer distintos nombres según versión; configuramos ambos.
os.environ["OPENAI_BASE_URL"] = base_url
os.environ["OPENAI_API_BASE"] = base_url
os.environ["OPENAI_API_KEY"] = github_token
os.environ["OPENAI_MODEL_NAME"] = "openai/gpt-4o"

# Verificar que las variables estén configuradas
print("🔍 Verificando configuración:")
print(f"OPENAI_BASE_URL: {os.environ.get('OPENAI_BASE_URL', 'No configurado')}")
print(f"GITHUB_TOKEN: {'✅ Configurado' if github_token else '❌ No configurado'}")
print(f"OPENAI_API_BASE: {os.environ.get('OPENAI_API_BASE', 'No configurado')}")
print(f"OPENAI_API_KEY: {'✅ Configurado' if os.environ.get('OPENAI_API_KEY') else '❌ No configurado'}")

if not github_token:
    print("\n⚠️ GITHUB_TOKEN no está configurado. Copia .env.example a .env y agrega tu token.")
else:
    print("\n✅ Variables de entorno mapeadas correctamente para CrewAI.")

🔍 Verificando configuración:
OPENAI_BASE_URL: https://models.inference.ai.azure.com
GITHUB_TOKEN: ✅ Configurado
OPENAI_API_BASE: https://models.inference.ai.azure.com
OPENAI_API_KEY: ✅ Configurado

✅ Variables de entorno mapeadas correctamente para CrewAI.


### 3. Configuración del LLM

Ahora configuramos el LLM usando la clase `LLM` de CrewAI. En las versiones actuales, `Agent` espera un `crewai.LLM` o un nombre de modelo, no un objeto `ChatOpenAI` de LangChain.

In [16]:
import re
import requests
from urllib.parse import quote
from crewai import LLM

WIKIPEDIA_API_URL = "https://es.wikipedia.org/w/api.php"
WIKIPEDIA_SUMMARY_URL = "https://es.wikipedia.org/api/rest_v1/page/summary"
WIKIPEDIA_HEADERS = {
    "User-Agent": "Curso-IA-DUOC/1.0 (notebook educativo; contacto: estudiante@example.com)"
}

# 🧠 Configurar el LLM con CrewAI
# Agent espera un crewai.LLM o un nombre de modelo; no acepta ChatOpenAI de LangChain.
try:
    if not github_token:
        raise EnvironmentError("GITHUB_TOKEN no configurado. Copia .env.example a .env y agrega tu token.")
    if not base_url:
        raise EnvironmentError("OPENAI_BASE_URL o GITHUB_BASE_URL no configurada en .env.")

    llm = LLM(
        model="openai/gpt-4o",
        base_url=base_url,
        api_key=github_token,
        temperature=0,
    )
    print("✅ LLM de CrewAI configurado correctamente.")
    
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    print("💡 Verifica que OPENAI_BASE_URL/GITHUB_BASE_URL y GITHUB_TOKEN estén configurados correctamente.")
    llm = None

✅ LLM de CrewAI configurado correctamente.


### 4. Definición de Herramientas con CrewAI

**⚠️ IMPORTANTE:** CrewAI requiere un enfoque específico para las herramientas.

**El Problema:**
- LangChain usa el decorador `@tool` para definir herramientas
- CrewAI requiere que las herramientas hereden de `BaseTool`
- Mezclar ambos enfoques causa errores

**La Solución:**
Usar `BaseTool` de `crewai.tools` para crear herramientas compatibles con CrewAI.

In [17]:
from crewai.tools import BaseTool


def _wikipedia_get_json(url, params=None):
    """Solicita JSON a Wikipedia y devuelve un error legible si la respuesta no es válida."""
    try:
        response = requests.get(
            url,
            params=params,
            headers=WIKIPEDIA_HEADERS,
            timeout=10,
        )
        response.raise_for_status()
        return response.json(), None
    except requests.exceptions.JSONDecodeError:
        return None, "Wikipedia devolvió una respuesta vacía o no válida en JSON."
    except requests.exceptions.RequestException as e:
        return None, f"No se pudo consultar Wikipedia: {e}"


def _limitar_oraciones(texto, max_oraciones=3):
    oraciones = re.split(r"(?<=[.!?])\s+", texto.strip())
    return " ".join(oraciones[:max_oraciones])


def buscar_resumen_wikipedia(query: str, max_oraciones=3) -> str:
    if not query or not query.strip():
        return "No se recibió un término de búsqueda para Wikipedia."

    search_data, error = _wikipedia_get_json(
        WIKIPEDIA_API_URL,
        params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": 1,
            "format": "json",
            "utf8": 1,
        },
    )
    if error:
        return error

    results = search_data.get("query", {}).get("search", [])
    if not results:
        return f"No se encontró ninguna página para '{query}'. Intenta con un término más específico."

    title = results[0]["title"]
    summary_data, error = _wikipedia_get_json(
        f"{WIKIPEDIA_SUMMARY_URL}/{quote(title)}"
    )
    if error:
        return error

    extract = summary_data.get("extract")
    if not extract:
        return f"Wikipedia no entregó un resumen disponible para '{title}'."

    return _limitar_oraciones(extract, max_oraciones=max_oraciones)


# 🔧 HERRAMIENTA CORREGIDA: Usar BaseTool en lugar de @tool
class WikipediaSearchTool(BaseTool):
    name: str = "Wikipedia Search Tool"
    description: str = "Busca en Wikipedia un tema y devuelve un resumen detallado. Es ideal para obtener información sobre personas, lugares, conceptos históricos y científicos."
    
    def _run(self, query: str) -> str:
        """Ejecuta la búsqueda en Wikipedia"""
        return buscar_resumen_wikipedia(query, max_oraciones=3)

# Crear la instancia de la herramienta
wikipedia_tool = WikipediaSearchTool()
tools = [wikipedia_tool]

# Probar la herramienta
try:
    test_result = wikipedia_tool._run("Albert Einstein")
    print("✅ Herramienta de Wikipedia configurada y probada exitosamente.")
    print(f"📝 Resultado de prueba: {test_result[:100]}...")
except Exception as e:
    print(f"❌ Error probando la herramienta: {e}")

print(f"\n🔧 {len(tools)} herramienta(s) disponible(s) para los agentes.")

✅ Herramienta de Wikipedia configurada y probada exitosamente.
📝 Resultado de prueba: Albert Einstein pronunciación en alemán: /ˈalbɐt ˈaɪnʃtaɪn/ (); fue un físico alemán de origen judío...

🔧 1 herramienta(s) disponible(s) para los agentes.


### 5. Creación del Equipo de Agentes (Crew)

Ahora definimos nuestro equipo de agentes especializados:

1. **Investigador (Researcher)**: Busca información detallada usando Wikipedia
2. **Escritor (Writer)**: Transforma la información en una biografía bien redactada

**⚠️ IMPORTANTE:** Otro error común es el parámetro `verbose` en `Crew`.

In [18]:
from crewai import Agent, Task, Crew, Process

# 🕵️ Agente 1: El Investigador
researcher = Agent(
    role="Investigador Senior",
    goal="Encontrar información completa y precisa sobre personas históricas, científicos y figuras importantes utilizando fuentes confiables.",
    backstory="""Eres un investigador académico con años de experiencia en la búsqueda de información histórica y científica. 
    Tu especialidad es encontrar datos precisos y relevantes en Wikipedia y otras fuentes confiables. 
    Te enorgulleces de la exactitud de tu trabajo y siempre proporcionas contexto histórico relevante. 
    No escribes biografías completas, tu trabajo es recopilar los datos más importantes y precisos.""",
    tools=tools,
    llm=llm,
    verbose=True,
    allow_delegation=False  # Este agente no delega trabajo
)

# ✍️ Agente 2: El Escritor
writer = Agent(
    role="Escritor de Biografías",
    goal="Crear biografías atractivas, bien estructuradas y fáciles de leer basadas en la información proporcionada por el investigador.",
    backstory="""Eres un escritor profesional especializado en biografías y divulgación científica. 
    Tu habilidad única es transformar datos técnicos y históricos en narrativas cautivadoras que son 
    tanto informativas como accesibles para el público general. 
    Tienes un don especial para destacar los aspectos más interesantes de la vida de las personas 
    y presentar sus logros de manera inspiradora.""",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("✅ Agentes creados exitosamente:")
print(f"🕵️ {researcher.role}")
print(f"✍️ {writer.role}")

✅ Agentes creados exitosamente:
🕵️ Investigador Senior
✍️ Escritor de Biografías


### 6. Definición de Tareas

Las tareas definen exactamente qué debe hacer cada agente y cómo se relacionan entre ellas.

In [19]:
# 🔍 Tarea 1: Investigación
research_task = Task(
    description="""Busca información detallada sobre Marie Curie en Wikipedia. 
    Enfócate en:
    - Sus descubrimientos científicos más importantes
    - Su impacto en la ciencia y la sociedad
    - Datos biográficos clave (fechas, lugares, educación)
    - Sus premios y reconocimientos
    - Su legado científico
    
    Proporciona información precisa y bien organizada que el escritor pueda usar.""",
    expected_output="Un resumen detallado de 4-6 párrafos con los datos más importantes sobre la vida, descubrimientos y legado de Marie Curie.",
    agent=researcher
)

# ✍️ Tarea 2: Escritura
write_task = Task(
    description="""Usando la información recopilada por el investigador, escribe una biografía cautivadora de Marie Curie.
    
    Requisitos:
    - Mínimo 5 párrafos bien estructurados
    - Estilo atractivo y accesible
    - Incluir sus logros más importantes
    - Destacar su impacto en la ciencia
    - Formato Markdown con encabezados apropiados
    - Tono inspirador pero preciso
    
    La biografía debe ser educativa e inspiradora para lectores de todas las edades.""",
    expected_output="Una biografía completa en formato Markdown, bien estructurada y atractiva, de al menos 5 párrafos.",
    agent=writer,
    context=[research_task]  # Esta tarea depende del resultado de la investigación
)

print("✅ Tareas definidas exitosamente:")
print(f"🔍 Tarea de investigación: {research_task.description[:50]}...")
print(f"✍️ Tarea de escritura: {write_task.description[:50]}...")

✅ Tareas definidas exitosamente:
🔍 Tarea de investigación: Busca información detallada sobre Marie Curie en W...
✍️ Tarea de escritura: Usando la información recopilada por el investigad...


### 7. Ensamblaje del Equipo (Crew)

**⚠️ CONFIGURACIÓN CORREGIDA:** El parámetro `verbose` debe ser boolean, no entero.

In [20]:
# 🎯 CONFIGURACIÓN CORREGIDA: verbose debe ser boolean
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,  # Las tareas se ejecutan en orden
    verbose=True  # ✅ CORRECTO: boolean, no entero (verbose=2 causaría error)
)

print("✅ Equipo (Crew) ensamblado exitosamente:")
print(f"👥 {len(crew.agents)} agentes en el equipo")
print(f"📋 {len(crew.tasks)} tareas definidas")
print(f"🔄 Proceso: {crew.process}")
print(f"🔊 Verbose: {crew.verbose}")

✅ Equipo (Crew) ensamblado exitosamente:
👥 2 agentes en el equipo
📋 2 tareas definidas
🔄 Proceso: Process.sequential
🔊 Verbose: True


### 8. Ejecución del Crew

¡Ahora viene la magia! Ejecutamos el crew y observamos cómo los agentes colaboran.

In [21]:
# 🚀 Ejecutar el crew
if llm is None:
    print("❌ No se puede ejecutar el crew sin un LLM configurado.")
    print("💡 Verifica la configuración de las variables de entorno en las celdas anteriores.")
else:
    try:
        print("🚀 Iniciando ejecución del crew...")
        print("📝 Observa cómo los agentes colaboran paso a paso:\n")
        
        # Ejecutar el crew
        result = crew.kickoff()
        
        print("\n" + "="*80)
        print("🏁 RESULTADO FINAL DEL CREW")
        print("="*80)
        print(result)
        
    except Exception as e:
        print(f"❌ Error durante la ejecución del crew: {e}")
        print("\n🔧 Posibles soluciones:")
        print("1. Verifica que GITHUB_TOKEN esté configurado correctamente")
        print("2. Asegúrate de que OPENAI_BASE_URL esté configurado")
        print("3. Confirma que tu token tenga permisos para usar GitHub Models")
        print("4. Verifica que no hayas mezclado @tool con BaseTool")
        print("5. Asegúrate de que verbose=True (no verbose=2)")
        
        # Mostrar información de debugging
        print("\n🐛 Información de debugging:")
        import traceback
        traceback.print_exc()

🚀 Iniciando ejecución del crew...
📝 Observa cómo los agentes colaboran paso a paso:



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a1874701-953e-4899-af1f-1bc3c71e23a8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Busca información detallada sobre Marie Curie en Wikipedia.                                              │
│      Enfócate en:                                                                                               │
│      - Sus descubrimientos científicos más importantes                                                          │
│      - Su impacto en la ciencia y la sociedad                                                                   │
│      - Datos biográficos clave (fechas, lugares, educación)                                                     │
│      - Sus premios y reconocimientos                                                                            │
│      - Su legado científico                                                                                     │
│                                                                                                                 │
│      Proporciona información precisa y bien organizada que el escritor pueda usar.                              │
│  ID: c865bc19-cf08-4208-aace-45af64ffc4c6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador Senior                                                                                     │
│                                                                                                                 │
│  Task: Busca información detallada sobre Marie Curie en Wikipedia.                                              │
│      Enfócate en:                                                                                               │
│      - Sus descubrimientos científicos más importantes                                                          │
│      - Su impacto en la ciencia y la sociedad                                                                   │
│      - Datos biográficos clave (fechas, lugares, educación)                                                     │
│      - Sus premios y reconocimientos                                                                            │
│      - Su legado científico                                                                                     │
│                                                                                                                 │
│      Proporciona información precisa y bien organizada que el escritor pueda usar.                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#6) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: wikipedia_search_tool                                                                                    │
│  Args: {'query': 'Marie Curie'}                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Usando la información recopilada por el investigador, escribe una biografía cautivadora de Marie Curie.  │
│                                                                                                                 │
│      Requisitos:                                                                                                │
│      - Mínimo 5 párrafos bien estructurados                                                                     │
│      - Estilo atractivo y accesible                                                                             │
│      - Incluir sus logros más importantes                                                                       │
│      - Destacar su impacto en la ciencia                                                                        │
│      - Formato Markdown con encabezados apropiados                                                              │
│      - Tono inspirador pero preciso                                                                             │
│                                                                                                                 │
│      La biografía debe ser educativa e inspiradora para lectores de todas las edades.                           │
│  Agent: Escritor de Biografías                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 
🏁 RESULTADO FINAL DEL CREW
# Marie Curie: La Pionera de la Ciencia Radiactiva

Marie Curie, nacida como Maria Salomea Skłodowska el 7 de noviembre de 1867 en Varsovia, Polonia, es una de las figuras más emblemáticas de la historia de la ciencia. Su vida y obra no solo revolucionaron el campo de la radiactividad, sino que también rompieron barreras de género en un mundo científico dominado por hombres. Con una mente brillante y una determinación inquebrantable, Curie se convirtió en la primera mujer en recibir un Premio Nobel y en la única persona en la historia en ganar este prestigioso galardón en dos disciplinas científicas diferentes: Física y Química.

## De Varsovia a París: El Inicio de una Leyenda

Desde joven, Marie mostró un talento excepcional para las ciencias, pero las limitaciones impuestas a las mujeres en su Polonia natal la llevaron a buscar oportunidades en el extranjero. En 1891, se trasladó a París p



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a1874701-953e-4899-af1f-1bc3c71e23a8                                                                       │
│  Final Output: # Marie Curie: La Pionera de la Ciencia Radiactiva                                               │
│                                                                                                                 │
│  Marie Curie, nacida como Maria Salomea Skłodowska el 7 de noviembre de 1867 en Varsovia, Polonia, es una de    │
│  las figuras más emblemáticas de la historia de la ciencia. Su vida y obra no solo revolucionaron el campo de   │
│  la radiactividad, sino que también rompieron barreras de género en un mundo científico dominado por hombres.   │
│  Con una mente brillante y una determinación inquebrantable, Curie se convirtió en la primera mujer en recibir  │
│  un Premio Nobel y en la única persona en la historia en ganar este prestigioso galardón en dos disciplinas     │
│  científicas diferentes: Física y Química.                                                                      │
│                                                                                                                 │
│  ## De Varsovia a París: El Inicio de una Leyenda                                                               │
│                                                                                                                 │
│  Desde joven, Marie mostró un talento excepcional para las ciencias, pero las limitaciones impuestas a las      │
│  mujeres en su Polonia natal la llevaron a buscar oportunidades en el extranjero. En 1891, se trasladó a París  │
│  para estudiar en la Universidad de la Sorbona, donde obtuvo títulos en Física y Matemáticas, a menudo          │
│  estudiando en condiciones de pobreza extrema. Fue en París donde conoció a Pierre Curie, un físico con quien   │
│  compartía no solo una pasión por la ciencia, sino también una visión de colaboración y descubrimiento.         │
│  Juntos, formaron una de las parejas científicas más influyentes de la historia.                                │
│                                                                                                                 │
│  ## El Descubrimiento del Polonio y el Radio                                                                    │
│                                                                                                                 │
│  En 1898, los Curie lograron un hito que cambiaría para siempre el curso de la ciencia: el descubrimiento de    │
│  dos nuevos elementos químicos, el polonio (nombrado en honor a Polonia) y el radio. Este trabajo fue           │
│  fundamental para el desarrollo del concepto de radiactividad, un término que Marie acuñó. En 1903, su          │
│  investigación sobre la radiación les valió a Marie y Pierre, junto con Henri Becquerel, el Premio Nobel de     │
│  Física. Este reconocimiento marcó un momento histórico, ya que Marie se convirtió en la primera mujer en       │
│  recibir un Nobel, desafiando las normas de su tiempo.                                                          │
│                                                                                                                 │
│  ## Un Segundo Nobel y un Legado Inmortal                                                                       │
│                                                                                                                 │
│  Tras la trágica muerte de Pierre en 1906, Marie conti

## 🎓 Resumen de Configuraciones Críticas

### Problemas Comunes y Sus Soluciones

| **Problema** | **Síntoma** | **Solución** |
|-------------|-------------|-------------|
| **Variables de entorno** | `AuthenticationError: Incorrect API key` | Mapear `GITHUB_TOKEN` → `OPENAI_API_KEY` y `OPENAI_BASE_URL`/`GITHUB_BASE_URL` → `OPENAI_API_BASE` |
| **LLM incompatible** | `ValidationError: llm.str / llm.is-instance[BaseLLM]` | Usar `crewai.LLM(...)` o un nombre de modelo, no `ChatOpenAI` de LangChain |
| **Herramientas incompatibles** | `'Tool' object is not callable` | Usar `BaseTool` en lugar de `@tool` |
| **Parámetro verbose** | `ValidationError: Input should be a valid boolean` | Usar `verbose=True` en lugar de `verbose=2` |
| **LLM no configurado** | `llm is None` | Verificar que las variables de entorno estén configuradas |

### Configuración Correcta para GitHub Models API

```python
from crewai import LLM
from crewai.tools import BaseTool

# 1. Mapear variables de entorno
base_url = os.environ.get("OPENAI_BASE_URL") or os.environ.get("GITHUB_BASE_URL", "")
github_token = os.environ.get("GITHUB_TOKEN", "")
os.environ["OPENAI_API_BASE"] = base_url
os.environ["OPENAI_API_KEY"] = github_token

# 2. Configurar LLM con la clase de CrewAI
llm = LLM(
    model="openai/gpt-4o",
    base_url=base_url,
    api_key=github_token,
    temperature=0,
)

# 3. Usar BaseTool para herramientas

class MiTool(BaseTool):
    name: str = "Mi Herramienta"
    description: str = "Descripción de la herramienta"
    def _run(self, query: str) -> str:
        return "resultado"

# 4. Configurar Crew con verbose boolean
crew = Crew(
    agents=[agent1, agent2],
    tasks=[task1, task2],
    process=Process.sequential,
    verbose=True  # ✅ boolean, no entero
)
```

### Variables de Entorno Requeridas

```bash
export OPENAI_BASE_URL="https://models.inference.ai.azure.com"
export GITHUB_TOKEN="tu_token_de_github_aquí"
```

## 🚀 Conclusiones

### Lo que hemos aprendido:

1. **CrewAI vs LangChain**: CrewAI orquesta equipos de agentes, LangChain proporciona los componentes individuales.

2. **Configuración crítica**: Cuando usas CrewAI con GitHub Models API, necesitas mapear las variables de entorno correctamente.

3. **Herramientas compatibles**: CrewAI requiere `BaseTool`, no el decorador `@tool` de LangChain.

4. **Parámetros correctos**: `verbose` debe ser boolean, no entero.

5. **Colaboración de agentes**: Los agentes pueden trabajar en secuencia, pasándose información entre ellos.

### Próximos pasos:

En los siguientes módulos exploraremos:
- Sistemas de memoria para agentes
- Integración con herramientas externas
- Estrategias de planificación más avanzadas
- Observabilidad y monitoreo de agentes

### 💡 Tip para estudiantes:

**Siempre revisa las configuraciones de entorno cuando cambies entre frameworks.** Cada framework tiene sus propias convenciones y expectativas sobre cómo acceder a los servicios externos. La clave está en entender estas diferencias y configurar correctamente las interfaces entre ellos.